In [41]:
import pandas as pd
dataset = pd.read_csv("CKD.csv")
dataset = pd.get_dummies(dataset, drop_first=True, dtype=int)
dataset

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2.000000,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,0,0,0,0,0,0,1,1,0,1
1,3.000000,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,1,0,0,0,0,0,1,0,0,1
2,4.000000,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,1,0,0,0,0,0,1,0,0,1
3,5.000000,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,1,0,0,0,0,0,1,0,1,1
4,5.000000,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,1,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,0.0,0.0,219.000000,36.000000,1.300000,139.000000,3.700000,12.500000,...,1,0,0,0,0,0,1,0,0,1
395,51.492308,70.000000,0.0,2.0,220.000000,68.000000,2.800000,137.528754,4.627244,8.700000,...,1,0,0,1,1,0,1,0,1,1
396,51.492308,70.000000,3.0,0.0,110.000000,115.000000,6.000000,134.000000,2.700000,9.100000,...,1,0,0,1,1,0,0,0,0,1
397,51.492308,90.000000,0.0,0.0,207.000000,80.000000,6.800000,142.000000,5.500000,8.500000,...,1,0,0,1,1,0,1,0,1,1


In [42]:
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [43]:
ind=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
dep = dataset[['classification_yes']]

In [44]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(ind, dep, test_size = 0.3 , random_state=0)

In [45]:
#from sklearn.preprocessing import StandardScaler
#scx = StandardScaler()
#X_train = scx.fit_transform(X_train)
#X_test = scx.transform(X_test)
# scy = StandardScaler()
# y_train = scy.fit_transform(y_train)
# y_test = scy.transform(y_test)

In [46]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

parm_grid = {'n_neighbors' : [3, 5, 2],
               'weights' : ['uniform', 'distance'],
               'algorithm' : ['auto' , 'ball_tree', 'kd_tree', 'brute'],
               'metric' : ['minkowski' , 'euclidean']}

from sklearn.neighbors import KNeighborsClassifier
grid = GridSearchCV(KNeighborsClassifier(), parm_grid, refit=True, verbose = 3,n_jobs=-1, cv=KFold(), scoring = 'f1_weighted')
grid.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


C:\Anaconda3\Lib\site-packages\sklearn\neighbors\_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


GridSearchCV(cv=KFold(n_splits=5, random_state=None, shuffle=False),
             estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'metric': ['minkowski', 'euclidean'],
                         'n_neighbors': [3, 5, 2],
                         'weights': ['uniform', 'distance']},
             scoring='f1_weighted', verbose=3)

In [47]:
grid_pred = grid.predict(X_test)

In [48]:
re = grid.cv_results_
table = pd.DataFrame.from_dict(re)
table.to_excel("KNN_Class_Grid.xlsx",index=False)

In [53]:
print("Evaluation metrics details for KNN are given below")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_pred)
print("1) Confusion Matrix:\n",cm)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_pred)
print('2) classification report:\n', clf_report)
from sklearn.metrics import roc_auc_score
ras = roc_auc_score(y_test, grid.predict_proba(X_test)[:,1])
print("3) roc_auc_score is", ras)
print("4) The best parameters from GridSearchCV for the created KNN model are : \n" , grid.best_params_)

Evaluation metrics details for KNN are given below
1) Confusion Matrix:
 [[39  6]
 [21 54]]
2) classification report:
               precision    recall  f1-score   support

           0       0.65      0.87      0.74        45
           1       0.90      0.72      0.80        75

    accuracy                           0.78       120
   macro avg       0.78      0.79      0.77       120
weighted avg       0.81      0.78      0.78       120

3) roc_auc_score is 0.8776296296296295
4) The best parameters from GridSearchCV for the created KNN model are : 
 {'algorithm': 'auto', 'metric': 'minkowski', 'n_neighbors': 2, 'weights': 'distance'}


In [50]:
#Optional step just to view the individual attributes from evaluation metrics
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test,grid_pred,average='macro')
f1_weighted = f1_score(y_test,grid_pred,average='weighted')
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test,grid_pred)
#to see the best parameters from GridSearchCV.best_params_ is one of teh attribute of GridSearchCV
print("The f1_weighted, f1_macro and accuracy values for the best parameters:", grid.best_params_, "are", f1_weighted ,",", f1_macro, "&", accuracy, "respectively")

The f1_weighted, f1_macro and accuracy values for the best parameters: {'algorithm': 'auto', 'metric': 'minkowski', 'n_neighbors': 2, 'weights': 'distance'} are 0.7785714285714286 , 0.7714285714285715 & 0.775 respectively


In [51]:
result = grid.predict([[2, 76.4599483204134, 3, 0, 148.112676056338, 57.4821052631579, 3.0773560209424, 137.52875399361, 4.62724358974359, 12.5181556195965, 38.8689024390243, 8408.19112627986, 4.70559701492537, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,1,1,0]])
result

C:\Anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


array([1])